# 🧪 Prefill and Decode Disaggregation Benchmark

Compare two ways of serving an LLM on the same endpoint.

**🟢 Baseline.** One vLLM engine does everything.

**🟣 Disaggregated.** One engine handles prefill (reading the prompt), another handles decode (generating tokens), connected by a router.

### What you will do

- ⚙️ Run the setup cells once (here in the notebook).

- 🟢 Start the **baseline** server in a terminal. Run the baseline cells in the notebook.

- ⛔ Stop it. 🟣 Start the **disaggregated** server in a terminal. Run the disagg cells in the notebook.

- 📊 Run the compare cell to see the plots.

### One rule

⚠️ Only one server runs at a time. Both bind port `8000`.

### Legend

- 🖥️ = run in a **terminal**
- 📓 = run in this **notebook**


## 🟢 ▶️ Start the baseline server

🖥️ **In a terminal:**

```bash
./start_baseline.sh
```

Wait until you see `Uvicorn running on http://0.0.0.0:8000`.

📓 Then run the next cells in this notebook.


## 🩺 Smoke test

📓 **Run in notebook.** Confirms the server on `http://localhost:8000` is alive.


In [ ]:
import requests

MODEL = "Qwen/Qwen2.5-7B-Instruct"
API_URL = "http://localhost:8000/v1/chat/completions"

res = requests.post(API_URL, json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Say hello"}],
    "max_tokens": 100,
})
res.raise_for_status()

print("Status:", res.status_code)
print("Latency:", res.elapsed.total_seconds(), "s")
print("Answer:", res.json()["choices"][0]["message"]["content"])


## ⚙️ Helpers — the experimental machinery

📓 **Run in notebook.** The next **three** code cells build the tools the experiment uses.

Each one is explained right above it. Run them once per kernel — no server calls happen yet.

### 🧱 Helper 1 of 3 — `make_unique_prompt`: defeat the prefix cache

vLLM caches the KV-tensor for prompt prefixes it has seen before. If we sent the same
prompt twice, the **second** call would skip prefill entirely and the TTFT would
collapse to ~0 — measuring **cache**, not **prefill**. That would make the disagg
vs. baseline comparison meaningless.

So every request gets a fresh, ~4000-word prompt:

```
┌─────────────────────────────────────────────────────────┐
│  "Session <uuid> nonce <64 random letters>. " ← header  │  ← unique every call,
│  "Discuss the CAP theorem ..."                          │    so vLLM cannot
│  "Compare leader-based and leaderless ..."              │    reuse any cached
│  "Explain how Paxos and Raft differ ..."                │    prefix.
│  ... (filler sentences until ~4000 words) ...           │
└─────────────────────────────────────────────────────────┘
```

- The UUID + random nonce at the front guarantees a **cache miss on token 1**.
- The long body forces a **real, heavy prefill** (lots of KV math) — which is
  the workload disaggregation is supposed to help with.



In [ ]:
import json
import random
import string
import threading
import time
import uuid

import requests

HEADERS = {"Content-Type": "application/json"}

FILLER_SENTENCES = [
    "Discuss the CAP theorem and its practical implications for modern databases.",
    "Compare leader-based and leaderless replication strategies in detail.",
    "Explain how Paxos and Raft differ in their handling of leader election.",
    "Describe the tradeoffs between strong, causal, and eventual consistency.",
    "Analyze how sharding interacts with cross-partition transactions.",
    "Outline the role of vector clocks in detecting concurrent updates.",
    "Examine how backpressure is implemented in streaming systems.",
    "Detail the lifecycle of a request through a service mesh.",
    "Contrast optimistic and pessimistic concurrency control mechanisms.",
    "Explore how exactly-once semantics can be achieved across services.",
]

def make_unique_prompt(target_words: int = 4000) -> str:
    """Long, unique prompt to defeat prefix caching and force real prefill."""
    header = f"Session {uuid.uuid4().hex} nonce {''.join(random.choices(string.ascii_lowercase, k=64))}. "
    body, words = [], 0
    while words < target_words:
        s = random.choice(FILLER_SENTENCES)
        body.append(s)
        words += len(s.split())
    return header + " ".join(body)


### 📏 Helper 2 of 3 — `run_request`: measure TTFT and ITL on one streamed call

This sends **one** chat completion with `stream=True` and times every chunk that
comes back over the wire. From the raw timestamps we derive the two metrics that
matter for an LLM-serving system:

```
       send                first                                       last
       prompt              token                                       token
         │                   │                                           │
   wall  ▼                   ▼     ▼   ▼     ▼   ▼   ▼   ▼   ▼   ▼   ▼   ▼
time ────●───────────────────●─────●───●─────●───●───●───●───●───●───●───●───►
         │←────── TTFT ─────→│←ITL→│←ITL→│←ITL→│  (gap between successive tokens)
         │                   │
         │ PREFILL phase     │     DECODE phase
         │ (read the prompt, │     (generate one token at a time,
         │  build KV cache)  │      reusing the KV cache)
```

- **TTFT** (Time To First Token) = how long until the model starts replying.
  Dominated by the **prefill** cost. This is what the disagg's dedicated prefill
  worker is supposed to keep snappy.

- **ITL** (Inter-Token Latency) = the gap between consecutive streamed tokens.
  Dominated by the **decode** cost. We report **p50** (typical) and **p95 / p99**
  (the slow tail). In a single-engine baseline, a fresh prefill from another user
  can pause your decode, spiking the tail. Disagg's dedicated decode worker
  should keep this **flat**.

To get byte-accurate first-token timing, we read the HTTP response with
`chunk_size=1` (the higher-level `iter_lines` would buffer and lie about TTFT).



In [ ]:
def _percentile(xs, p):
    if not xs:
        return None
    xs = sorted(xs)
    k = max(0, min(len(xs) - 1, int(round((p / 100.0) * (len(xs) - 1)))))
    return xs[k]


def run_request(results, name, prompt=None, max_tokens=400):
    """Stream one chat completion; record TTFT, total time, and ITL stats."""
    if prompt is None:
        prompt = make_unique_prompt()

    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True,
        "cache_prompt": False,
    }

    start = time.time()
    first_token = None
    event_times = []  # timestamp of every SSE chunk -> tokens

    with requests.post(API_URL, headers=HEADERS,
                       data=json.dumps(payload), stream=True) as r:
        if r.status_code != 200:
            raise RuntimeError(f"{name}: HTTP {r.status_code} {r.text[:200]}")

        buf = b""
        # chunk_size=1 -> byte-accurate TTFT (iter_lines buffers)
        for chunk in r.iter_content(chunk_size=1, decode_unicode=False):
            if not chunk:
                continue
            if first_token is None:
                first_token = time.time()
            buf += chunk
            while b"\n\n" in buf:
                event, buf = buf.split(b"\n\n", 1)
                if event.startswith(b"data:") and b"[DONE]" not in event:
                    event_times.append(time.time())

    end = time.time()

    # ITL = gap between successive decode events
    itls = [t2 - t1 for t1, t2 in zip(event_times, event_times[1:]) if (t2 - t1) > 0]

    results[name] = {
        "ttft": first_token - start if first_token else None,
        "total": end - start,
        "prompt_chars": len(prompt),
        "tokens": len(event_times),
        "itl_mean": (sum(itls) / len(itls)) if itls else None,
        "itl_p50": _percentile(itls, 50),
        "itl_p95": _percentile(itls, 95),
        "itl_p99": _percentile(itls, 99),
    }


### 🌀 Helper 3 of 3 — `sweep`: fire N requests in parallel, escalate the load

A single-request test is boring — both systems will look fine. The interesting
question is **what happens under concurrency**, where a fresh prefill from
user B can stall the decode stream of user A.

`run_parallel(n)` launches **n** threads that hit the endpoint at the same
moment, each with its own unique 4000-word prompt:

```
            ┌───────────────────────────► server (port 8000)
   t=0  ────┼───────────────────────────►
            ├───────────────────────────►
            └───────────────────────────►
            (n requests in flight at once)
```

`sweep(...)` repeats this at **n = 1, 2, 4, 8** and collects per-concurrency
stats. The intuition for the two systems:

```
   BASELINE (one engine does both)        DISAGG (split workers)
   ────────────────────────────────       ────────────────────────────────
   GPU0: [prefill A][decode A]            GPU0 (prefill): [A][B][C][D]...
   GPU0: [prefill B][decode B]                                  ▼ KV transfer
         ↑ each new prefill freezes       GPU1 (decode):  [A][B][C][D]...
           every ongoing decode                            ↑ never pauses for prefill
           → ITL p95 spikes                                  → ITL p95 stays flat
```

`summarise(...)` averages TTFT and ITL percentiles **across the n parallel
requests**, then `print_sweep_table(...)` prints a tidy row per concurrency
level so you can eyeball the scaling shape.



In [ ]:
import statistics


def _ms(x):
    return f"{x*1000:>5.0f}ms" if x is not None else "    -  "


def run_parallel(n=3, verbose=True):
    """Fire n requests in parallel, each with its own unique prompt."""
    results = {}
    prompts = [make_unique_prompt() for _ in range(n)]
    threads = [
        threading.Thread(target=run_request, args=(results, f"req_{i}"),
                         kwargs={"prompt": prompts[i]})
        for i in range(n)
    ]
    start = time.time()
    for t in threads: t.start()
    for t in threads: t.join()
    wall = time.time() - start

    if verbose:
        for k, v in sorted(results.items()):
            print(f"  {k:6s}  TTFT {v['ttft']:5.2f}s  Total {v['total']:5.2f}s  "
                  f"tokens {v['tokens']:>3d}  "
                  f"ITL p50 {_ms(v['itl_p50'])} p95 {_ms(v['itl_p95'])} p99 {_ms(v['itl_p99'])}")
        print(f"  wall {wall:.2f}s")
    return results, wall


def summarise(results):
    ttfts = [v["ttft"]    for v in results.values() if v["ttft"]    is not None]
    p50s  = [v["itl_p50"] for v in results.values() if v["itl_p50"] is not None]
    p95s  = [v["itl_p95"] for v in results.values() if v["itl_p95"] is not None]
    p99s  = [v["itl_p99"] for v in results.values() if v["itl_p99"] is not None]
    return {
        "ttft_mean":    statistics.mean(ttfts) if ttfts else None,
        "ttft_p95":     sorted(ttfts)[int(0.95 * (len(ttfts) - 1))] if ttfts else None,
        "itl_p50_mean": statistics.mean(p50s)  if p50s  else None,
        "itl_p95_mean": statistics.mean(p95s)  if p95s  else None,
        "itl_p99_mean": statistics.mean(p99s)  if p99s  else None,
    }


def sweep(concurrencies=(1, 2, 4, 8)):
    rows = []
    for n in concurrencies:
        print(f"\nconcurrency = {n}")
        results, wall = run_parallel(n)
        row = summarise(results) | {"concurrency": n, "wall": wall}
        rows.append(row)
    return rows


def print_sweep_table(rows):
    print(f"\n{'conc':>4} {'TTFT mean':>10} {'TTFT p95':>10} "
          f"{'ITL p50':>9} {'ITL p95':>9} {'ITL p99':>9} {'wall':>8}")
    for r in rows:
        def fmt(x, unit="s"):
            if x is None: return "    -  "
            return f"{x*1000:7.0f}ms" if unit == "ms" else f"{x:7.2f}s"
        print(f"{r['concurrency']:>4} {fmt(r['ttft_mean']):>10} {fmt(r['ttft_p95']):>10} "
              f"{fmt(r['itl_p50_mean'],'ms'):>9} {fmt(r['itl_p95_mean'],'ms'):>9} "
              f"{fmt(r['itl_p99_mean'],'ms'):>9} {fmt(r['wall']):>8}")


---

## 🟢 Scenario A — Baseline

🖥️ **Make sure the baseline server is still running** (from the start cell above).

### What this experiment does

You are about to send the same workload at the **single-engine** vLLM and watch
how it copes as the load goes up.

```
   you ──► localhost:8000 ──► ┌──────────────────────────────┐
                              │  vLLM (one engine, GPU 0)    │
                              │  prefill + decode in one box │
                              └──────────────────────────────┘
```

Two cells, in order:

1. **Warmup.** One throwaway request with a *unique* prompt (so it does **not**
   pollute the prefix cache). This pays the JIT / CUDA-graph / autotuner cost
   so the real measurements don't get a misleading slow first request.

2. **Sweep.** `sweep((1, 2, 4, 8))` — fire 1, then 2, then 4, then 8
   concurrent requests, each a fresh 4000-word prompt. For every level we
   record mean TTFT and ITL p50 / p95 / p99 across the parallel requests.

The numbers land in `baseline_sweep` for the comparison plot at the end.

📓 Run the next two cells now.



In [ ]:
# Warmup with a throwaway unique prompt (does not pollute the prefix cache).
import json

print("Warmup (baseline)...")
_w = {}
run_request(_w, "warmup", prompt=make_unique_prompt(target_words=500))
print(json.dumps(_w["warmup"], indent=2, default=float))


In [ ]:
# Concurrency sweep against the BASELINE endpoint.
# Saves into `baseline_sweep` for the comparison plot at the end.
print("=== BASELINE concurrency sweep ===")
baseline_sweep = sweep((1, 2, 4, 8))

print("\nSummary:")
print_sweep_table(baseline_sweep)


---

## 🟣 Scenario B — Disaggregated

🖥️ **In a terminal — switch servers:**

```bash
# Stop the baseline first (Ctrl+C in its terminal), then:
./start_disagg.sh
```

Wait until you see `Disaggregated endpoint is now live at http://localhost:8000/...`.

### What this experiment does

Exactly the **same** workload as Scenario A — same prompts, same warmup, same
`sweep((1, 2, 4, 8))` — but now hitting the **split** stack:

```
   you ──► localhost:8000 (router)
                │
                ├──► ┌─────────────────────────┐
                │    │ prefill vLLM, GPU 0     │  ── builds KV cache
                │    └──────────┬──────────────┘
                │               │ KV transfer (NIXL)
                │               ▼
                └──► ┌─────────────────────────┐
                     │ decode  vLLM, GPU 1     │  ── streams tokens
                     └─────────────────────────┘
```

Because decode lives on its own GPU, a fresh prefill from a new user **cannot**
pre-empt a decode that's already in flight. The hypothesis we are testing:

> Under concurrency, **ITL p95 / p99 stay flatter** on disagg than on baseline,
> even if absolute TTFT is no better (the prefill GPU is doing the same work
> alone that the baseline was doing alongside decode).

Results land in `disagg_sweep`. 📓 Run the next two cells.



In [ ]:
print("Warmup (disagg)...")
_w = {}
run_request(_w, "warmup", prompt=make_unique_prompt(target_words=500))
print(json.dumps(_w["warmup"], indent=2, default=float))


In [ ]:
# Concurrency sweep against the DISAGGREGATED endpoint.
print("=== DISAGG concurrency sweep ===")
disagg_sweep = sweep((1, 2, 4, 8))

print("\nSummary:")
print_sweep_table(disagg_sweep)


---

## 📊 Compare — does disagg actually win?

📓 **Run in notebook.** Needs both `baseline_sweep` and `disagg_sweep` to exist.

This cell plots the two sweeps side by side, one panel per metric, **x-axis =
concurrency (1, 2, 4, 8)**:

```
   ▲ TTFT (s)          ▲ ITL p50 (ms)        ▲ ITL p95 (ms)
   │   baseline ●─●─●  │  baseline ●─●─●     │  baseline    ●
   │                   │                     │           ●
   │   disagg   ■─■─■  │  disagg   ■─■─■     │        ●
   │                   │                     │     ●
   │                   │                     │  disagg ■─■─■─■  ← the win
   └──────────► conc   └──────────► conc     └──────────► conc
```

### How to read it

- **TTFT panel** — *"how fast does a reply start?"*
  Expect baseline and disagg to be **roughly comparable** at low concurrency.
  At high concurrency disagg may even be slightly worse (its prefill GPU is
  alone instead of helping with both phases).

- **ITL p50 panel** — *"typical token gap"*
  Both should be similar. The median user usually doesn't notice prefill
  interference.

- **ITL p95 panel — the headline result.** *"slow-tail token gap"*
  This is where disagg is **supposed** to win: baseline climbs as concurrency
  rises (decodes get paused by new prefills), disagg stays roughly flat
  (decode GPU is never interrupted). If you see the baseline curve sloping up
  and the disagg curve staying flat, the experiment confirmed the hypothesis.



In [ ]:
import json
import matplotlib.pyplot as plt

assert "baseline_sweep" in dir(), "Run the Baseline scenario first."
assert "disagg_sweep"   in dir(), "Run the Disagg scenario first."

# Side-by-side numbers, neatly formatted.
print("Baseline:")
print(json.dumps(baseline_sweep, indent=2, default=float))
print("\nDisagg:")
print(json.dumps(disagg_sweep, indent=2, default=float))

# Plot.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plots = [
    (axes[0], "ttft_mean",    "TTFT (mean)",    "s"),
    (axes[1], "itl_p50_mean", "ITL p50",        "ms"),
    (axes[2], "itl_p95_mean", "ITL p95 (tail)", "ms"),
]
for ax, key, title, unit in plots:
    for label, rows, marker in [("baseline", baseline_sweep, "o"),
                                ("disagg",   disagg_sweep,   "s")]:
        xs = [r["concurrency"] for r in rows]
        ys = [r[key] for r in rows]
        if unit == "ms":
            ys = [(y * 1000) if y is not None else None for y in ys]
        ax.plot(xs, ys, marker=marker, label=label)
    ax.set_xlabel("concurrent requests")
    ax.set_ylabel(unit)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()
